<a href="https://colab.research.google.com/github/Revanthpendyala02/agentic-ai-lab/blob/master/Assignment_4_Multi_Agent_Collaboration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain-google-genai ddgs

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.5/571.5 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 13.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.1 which is incompatible.


In [2]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get("gem")

print("API key loaded:", GEMINI_API_KEY is not None)

API key loaded: True


In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    google_api_key=GEMINI_API_KEY
)

print("Gemini connected successfully!")

Gemini connected successfully!


In [4]:
from ddgs import DDGS

def web_search(query, max_results=3):
    results = []

    with DDGS() as ddgs:
        for result in ddgs.text(query, max_results=max_results):
            results.append({
                "title": result["title"],
                "url": result["href"],
                "content": result["body"]
            })

    return results

print("Web search function ready!")

Web search function ready!


In [5]:
def research_agent(topic):

    results = web_search(topic, 3)

    research_data = ""

    for i, result in enumerate(results, 1):
        research_data += f"""
Source {i}:
Title: {result['title']}
Content: {result['content']}
"""

    prompt = f"""
You are the Research Agent.

Research Topic: {topic}

Based on the sources below, provide:
- Definition
- Key concepts
- Applications
- Advantages
- Limitations

Be concise and do not invent information.

Sources:
{research_data}
"""

    response = llm.invoke(prompt)

    content = response.content

    if isinstance(content, list):
        content = "\n".join(
            item.get("text", "")
            for item in content
            if isinstance(item, dict)
        )

    return content, results

In [6]:
def analyst_agent(research):

    prompt = f"""
You are the Analyst Agent.

Analyze the following research:

{research}

Provide:
1. Key findings
2. Important benefits
3. Major limitations
4. Practical applications
5. Overall analysis

Keep the response concise.
"""

    response = llm.invoke(prompt)

    content = response.content

    if isinstance(content, list):
        content = "\n".join(
            item.get("text", "")
            for item in content
            if isinstance(item, dict)
        )

    return content

In [7]:
def report_agent(topic, research, analysis):

    prompt = f"""
You are the Report Agent.

Create a professional report on:

{topic}

Research:
{research}

Analysis:
{analysis}

Use this structure:

# {topic}

## 1. Introduction
## 2. Definition
## 3. Key Concepts
## 4. Applications
## 5. Advantages
## 6. Limitations
## 7. Analysis
## 8. Conclusion

Keep it clear and concise.
"""

    response = llm.invoke(prompt)

    content = response.content

    if isinstance(content, list):
        content = "\n".join(
            item.get("text", "")
            for item in content
            if isinstance(item, dict)
        )

    return content

In [8]:
topic = "Agentic AI"

research, sources = research_agent(topic)

analysis = analyst_agent(research)

final_report = report_agent(
    topic,
    research,
    analysis
)

print("=" * 60)
print("MULTI-AGENT FINAL REPORT")
print("=" * 60)
print(final_report)

MULTI-AGENT FINAL REPORT
# Report: Agentic AI

## 1. Introduction
The landscape of artificial intelligence is shifting from static, prompt-response interactions toward dynamic, action-oriented systems. Agentic AI represents the next frontier of this evolution, transforming AI from a passive information assistant into an active participant capable of executing complex workflows. This report outlines the architecture, utility, and current limitations of Agentic AI.

## 2. Definition
Agentic AI refers to intelligent systems designed to pursue goals, utilize external tools, and execute multi-step processes with a degree of operational autonomy. Unlike traditional chatbots, which function primarily as information retrieval engines, Agentic AI acts as an orchestrator, navigating software environments to achieve specific end states.

## 3. Key Concepts
*   **Goal-Directed Behavior:** The ability to decompose complex objectives into actionable steps.
*   **System Architecture:** A framework co

In [9]:
print("=" * 60)
print("AGENT COLLABORATION")
print("=" * 60)

print("\n[1] RESEARCH AGENT")
print(research)

print("\n[2] ANALYST AGENT")
print(analysis)

print("\n[3] REPORT AGENT")
print(final_report)

AGENT COLLABORATION

[1] RESEARCH AGENT
As the Research Agent, I have synthesized the information from the provided sources regarding Agentic AI:

### **Definition**
Agentic AI refers to artificial intelligence programs capable of pursuing goals, utilizing external tools, and executing actions with a degree of autonomy. Unlike traditional "tool-like" AI or basic chatbots designed for narrow tasks, agentic AI is characterized by its ability to perform multi-step processes.

### **Key Concepts**
*   **Autonomy:** The capacity to operate with self-directed initiative rather than functioning strictly as a passive response tool.
*   **Control Flow:** Systems are frequently powered by Large Language Models (LLMs) that provide the intelligence behind the agent’s logic.
*   **System Architecture:** Agentic systems combine multiple components, including planning logic, memory, tool interfaces, and orchestration software to coordinate tasks.
*   **Goal-Directed Behavior:** The system is designed

In [10]:
topic = input("Enter research topic: ")

research, sources = research_agent(topic)

analysis = analyst_agent(research)

final_report = report_agent(
    topic,
    research,
    analysis
)

print("\n" + "=" * 60)
print("FINAL REPORT")
print("=" * 60)
print(final_report)

Enter research topic: dna

FINAL REPORT
# DNA

## 1. Introduction
Deoxyribonucleic acid (DNA) is the fundamental molecule of life, serving as the universal blueprint for the development, function, and reproduction of all known living organisms. As the primary repository of genetic information, DNA bridges the gap between molecular chemistry and biological complexity.

## 2. Definition
DNA is a polymer composed of two polynucleotide chains that coil around each other to form a double helix. It is one of the four major macromolecules essential for life, functioning as a high-density, stable medium for storing biological data.

## 3. Key Concepts
*   **Molecular Composition:** Each nucleotide consists of a sugar (deoxyribose), a phosphate group, and one of four nitrogenous bases: Adenine (A), Thymine (T), Cytosine (C), or Guanine (G).
*   **Base Pairing:** Complementary pairing (A with T, C with G) is held together by hydrogen bonds, forming the structural basis of the helix.
*   **Organi